In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from huggingface_hub import HfFileSystem
import polars as pl

fs = HfFileSystem()

number_of_files = 1
event_type = 'ttbar_pu0'
number_of_hf_repo_files= 100
# Load particles
particles_list = []
for i in range(number_of_files):
    file_path = f"datasets/CERN/ColliderML-Release-1/data/{event_type}_particles/train-{i:05d}-of-{number_of_hf_repo_files:05d}.parquet"
    with fs.open(file_path, "rb") as f:
        particles_list.append(pl.read_parquet(f))
particles = pl.concat(particles_list)

# Load calo_hits
calo_hits_list = []
for i in range(number_of_files):
    file_path = f"datasets/CERN/ColliderML-Release-1/data/{event_type}_calo_hits/train-{i:05d}-of-{number_of_hf_repo_files:05d}.parquet"
    with fs.open(file_path, "rb") as f:
        calo_hits_list.append(pl.read_parquet(f))
calo_hits = pl.concat(calo_hits_list)

# Load tracks
tracks_list = []
for i in range(number_of_files):
    file_path = f"datasets/CERN/ColliderML-Release-1/data/{event_type}_tracks/train-{i:05d}-of-{number_of_hf_repo_files:05d}.parquet"
    with fs.open(file_path, "rb") as f:
        tracks_list.append(pl.read_parquet(f))
tracks = pl.concat(tracks_list)

In [14]:
event_id = 377
particle_id = 145

In [15]:
(particles.lazy()
 .select(['event_id', 'particle_id', 'pdg_id', 'vx', 'vy', 'vz', 'energy'])
 .explode('particle_id', 'pdg_id', 'vx', 'vy', 'vz', 'energy')
.filter((pl.col('event_id')==event_id) & (pl.col('particle_id')==particle_id))   
 ).collect()

event_id,particle_id,pdg_id,vx,vy,vz,energy
u32,u64,i64,f32,f32,f32,f32
377,145,111,-0.000826,0.005181,8.027347,19.570423


In [11]:
(calo_hits.lazy()
    .select(['event_id', 'contrib_particle_ids','contrib_energies', 'x', 'y', 'z','detector'])
    .explode('contrib_particle_ids', 'contrib_energies', 'x', 'y', 'z', 'detector')
    .explode('contrib_particle_ids', 'contrib_energies') # Double explode if list[list]
    .rename({'contrib_particle_ids': 'particle_id', 'contrib_energies': 'energy_contribution'})
    .filter((pl.col('event_id')==441) & (pl.col('particle_id')==611))
    
    ).collect()

event_id,particle_id,energy_contribution,x,y,z,detector
u32,u64,f32,f32,f32,f32,u8
441,611,0.000262,486.470306,306.386414,-3232.699951,9
441,611,0.000165,484.518616,311.098175,-3283.199951,9
441,611,0.00039,475.095062,307.194824,-3212.5,9
441,611,0.000123,479.806824,309.146515,-3217.550049,9
441,611,0.000242,462.015778,332.190948,-3202.399902,9


In [17]:
(particles.lazy()
 .select(['event_id','particle_id', 'parent_id', 'pdg_id', 'vx', 'vy', 'vz', 'energy'])
 .explode('particle_id','pdg_id', 'parent_id', 'vx', 'vy', 'vz', 'energy')
.filter((pl.col('event_id')==event_id) & (pl.col('parent_id')==particle_id))
).collect()

event_id,particle_id,parent_id,pdg_id,vx,vy,vz,energy
u32,u64,i64,i64,f32,f32,f32,f32
377,2768,145,22,-0.000706,0.004854,8.028852,19.49156
377,2769,145,11,-945.5401,391.170197,228.582809,0.000771
